# 手写 Multi-Head Attention (MHA)

## 1. 核心公式

### Single-Head: Scaled Dot-Product Attention

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

### Multi-Head Attention

$$\text{MultiHead}(X) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h) \cdot W_O$$

其中每个 head：

$$\text{head}_i = \text{Attention}(XW_Q^{(i)},\; XW_K^{(i)},\; XW_V^{(i)})$$

### 分步理解

1. 用 $W_Q, W_K, W_V$ 把输入 $X$ 映射到 Q, K, V（维度 = $d_{\text{model}}$）
2. 把 Q, K, V **拆成 $h$ 个 head**（每个 head 的维度 $d_k = d_{\text{model}} / h$）
3. 每个 head **独立**做 Scaled Dot-Product Attention
4. 把 $h$ 个 head 的输出 **concat** 回 $d_{\text{model}}$ 维度
5. 过一个输出投影 $W_O$

### 关键维度关系

$$d_k = d_{\text{model}} / h$$

例如 GPT-2: $d_{\text{model}} = 768$, $h = 12$, $d_k = 64$

## 2. 代码变量与公式元素对应

| 代码变量 | 公式符号 | 含义 | shape |
|----------|----------|------|-------|
| `x` | $X$ | 输入序列 | `(B, T, D)` |
| `self.W_q` | $W_Q$ | Query 投影 | `(D, D)` |
| `self.W_k` | $W_K$ | Key 投影 | `(D, D)` |
| `self.W_v` | $W_V$ | Value 投影 | `(D, D)` |
| `self.W_o` | $W_O$ | 输出投影 | `(D, D)` |
| `Q, K, V` (投影后) | $XW_Q$ 等 | 投影结果 | `(B, T, D)` |
| `Q, K, V` (拆 head 后) | 各 head | 每个 head 的 Q/K/V | `(B, h, T, d_k)` |
| `scores` | $QK^T / \sqrt{d_k}$ | 注意力分数 | `(B, h, T, T)` |
| `attn_weights` | $\text{softmax}(\cdot)$ | 注意力权重 | `(B, h, T, T)` |
| `attn_output` | $\text{head}_i$ → concat | 加权求和后合并 | `(B, T, D)` |
| `output` | $\cdot W_O$ | 最终输出 | `(B, T, D)` |

> **B** = batch, **T** = seq_len, **D** = d_model, **h** = n_heads, **d_k** = D // h

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

## 3. 完整实现

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads

        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, mask=None):
        # x: (B, T, d_model),  mask: (1, 1, T, T) 或 (B, 1, 1, T)
        # batch size, Sequence Length, embedding dimension
        B, T, _ = x.shape

        # 1. 线性投影
        Q = self.W_q(x)  # (B, T, d_model)
        K = self.W_k(x)
        V = self.W_v(x)

        # 2. 拆 head: (B, T, d_model) -> (B, h, T, d_k)
        Q = Q.view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        K = K.view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        V = V.view(B, T, self.n_heads, self.d_k).transpose(1, 2)

        # 3. 注意力分数: (B, h, T, T)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)

        # 4. Mask（causal / padding）
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))

        # 5. Softmax
        attn_weights = F.softmax(scores, dim=-1)  # (B, h, T, T)

        # 6. 加权求和: (B, h, T, d_k)
        attn_output = torch.matmul(attn_weights, V)

        # 7. 合并 head: (B, T, d_model)
        attn_output = attn_output.transpose(1, 2).contiguous().view(B, T, self.d_model)

        # 8. 输出投影
        output = self.W_o(attn_output)  # (B, T, d_model)
        return output

## 4. Shape 变化全流程图

```
x: (B, T, d_model)                    例: (2, 5, 64)
        | W_q / W_k / W_v
Q, K, V: (B, T, d_model)              例: (2, 5, 64)
        | view + transpose (拆 head)
Q, K, V: (B, h, T, d_k)              例: (2, 8, 5, 8)
        | Q @ K^T / sqrt(d_k)
scores: (B, h, T, T)                  例: (2, 8, 5, 5)
        | mask + softmax
attn_weights: (B, h, T, T)            例: (2, 8, 5, 5)
        | @ V
attn_output: (B, h, T, d_k)          例: (2, 8, 5, 8)
        | transpose + view (合并 head)
attn_output: (B, T, d_model)          例: (2, 5, 64)
        | W_o
output: (B, T, d_model)               例: (2, 5, 64)
```

## 5. 测试

In [ ]:
batch_size, seq_len, d_model, n_heads = 2, 5, 64, 8
x = torch.randn(batch_size, seq_len, d_model)

mha = MultiHeadAttention(d_model, n_heads)
output = mha(x)
print(f"输入 shape: {x.shape}")      # (2, 5, 64)
print(f"输出 shape: {output.shape}")  # (2, 5, 64)

In [ ]:
# Causal Mask 测试
causal_mask = torch.tril(torch.ones(seq_len, seq_len)).view(1, 1, seq_len, seq_len)
print("Causal Mask:")
print(causal_mask[0, 0])

output_masked = mha(x, mask=causal_mask)
print(f"带 mask 输出 shape: {output_masked.shape}")  # (2, 5, 64)

## 6. 常见面试追问

| 问题 | 要点 |
|------|------|
| 为什么除以 $\sqrt{d_k}$？ | $QK^T$ 方差 $\approx d_k$，不缩放则 softmax 饱和、梯度消失 |
| 为什么用多个 head？ | 不同 head 学不同注意力模式，总计算量不变 |
| `W_q` 为什么是 `(d_model, d_model)` 不是 `(d_model, d_k)`？ | 一个大矩阵同时算所有 head 的 Q，等价于 h 个小矩阵但更高效 |
| `.contiguous()` 干什么？ | `transpose` 后内存不连续，`view` 要求连续，所以需要 `contiguous()` |
| MHA vs MQA vs GQA？ | MHA: 每个 head 独立 K/V；MQA: 所有 head 共享 K/V；GQA: 分组共享（折中） |
| Causal Mask shape `(1,1,T,T)` 为什么？ | 两个 1 分别广播到 batch 和 n_heads |
| $W_O$ 的作用？ | 让不同 head 的信息融合交互 |
| `masked_fill(-inf)` 为什么有效？ | $e^{-\infty}=0$，softmax 后该位置权重为 0 |

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model%n_heads==0
        self.d_k=d_model//n_heads

        self.d_model=d_model
        self.num_heads=n_heads
        
        self.Wq=nn.Linear(d_model,d_model,bias=False)
        self.Wk=nn.Linear(d_model,d_model,bias=False)
        self.Wv=nn.Linear(d_model,d_model,bias=False)
        self.Wo=nn.Linear(d_model,d_model,bias=False)

    def forward(self, x, is_causal=False):
        B, T, _ = x.shape
        Q = self.Wq(x)
        K = self.Wk(x)
        V = self.Wv(x)

        Q = Q.view(B, T, self.num_heads, self.d_k).transpose(1, 2)
        K = K.view(B, T, self.num_heads, self.d_k).transpose(1, 2)
        V = V.view(B, T, self.num_heads, self.d_k).transpose(1, 2)

        attention = torch.matmul(Q, K.transpose(2, 3)) / math.sqrt(self.d_k)

        if is_causal:
            mask = torch.tril(torch.ones(T, T, device=x.device)).bool()
            attention = attention.masked_fill(~mask, float('-inf'))

        attention_weights = F.softmax(attention, dim=-1)

        res = torch.matmul(attention_weights, V)
        res = res.transpose(1, 2).contiguous().view(B, T, self.d_model)
        return self.Wo(res)